In [1]:
# =============================================================================
# 03_llm_strategy_evaluation.ipynb
# =============================================================================
# Purpose:
#   Evaluate the LLM-generated consulting strategies that sit on top of DiCE
#   counterfactuals, focusing on *faithfulness* rather than predictive accuracy.
#   Three questions are addressed:
#     (A) Input grounding - what fraction of the counterfactual signal that the
#         LLM is asked to translate is actionable vs. non-controllable?
#     (B) Offline text metrics on the delivered strategies - grounding coverage,
#         non-actionable leakage, and structural completeness.
#     (C) A reproducible, API-gated harness for LLM-as-judge rubric scoring and
#         prompt/model ablation (runs when OPENAI_API_KEY is available).
#   A confidence-gating analysis quantifies how many instances are eligible for
#   trustworthy strategy generation, mitigating the circular dependence of the
#   advice on a low-accuracy predictor.
#
# Inputs:
#   ../data/selected_features.pkl, ../data/modeling_dataset_top30.csv
#   ../data/original_outputs/consulting_case{1,2}_strategy.txt
#   ../data/original_outputs/consulting_case{1,2}_changes.csv
#   ../results/tables/cf_generated_sample.csv            (from notebook 02)
# Outputs (results/):
#   tables/t6_strategy_faithfulness.csv
#   tables/t7_confidence_gating.csv
#   tables/t8_input_actionability_by_regime.csv
#   figures/f7_strategy_faithfulness.(png|pdf)
#   figures/f8_confidence_gating.(png|pdf)
# =============================================================================

import os, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import xgboost as xgb

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.size": 11, "axes.edgecolor": "black", "axes.linewidth": 0.8,
                     "savefig.dpi": 600, "figure.dpi": 120, "pdf.fonttype": 42, "ps.fonttype": 42})
GRAY_DARK, GRAY_MID, GRAY_LIGHT = "0.20", "0.45", "0.75"
FIG_DIR = Path("../results/figures"); FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR = Path("../results/tables");  TAB_DIR.mkdir(parents=True, exist_ok=True)
def save_fig(fig, name):
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{name}.pdf", bbox_inches="tight")
    plt.close(fig); print(f"  saved figure: {name}.png / {name}.pdf")

features = joblib.load("../data/selected_features.pkl")
ACTIONABLE = [f for f in features if "Sales" in f]
IMMUTABLE  = [f for f in features if "Population" in f]

# Three-tier actionability taxonomy (refined per referee comment). The available
# administrative data contain no DIRECT control variables (menu, price, hours,
# staffing); the estimated-sales features are MEDIATING OUTCOMES an owner can
# influence but not set directly, and the population features are EXOGENOUS
# district context. The constraint operates on the mediating-outcome tier versus
# the exogenous tier; the absence of a direct-control tier is a data limitation
# we state explicitly rather than paper over.
TIER = {}
for f in features:
    TIER[f] = "mediating_outcome" if "Sales" in f else "exogenous_context"
DIRECT_CONTROL = []   # not represented in the administrative feature set
INFLUENCEABLE = ACTIONABLE   # == mediating-outcome tier (what the owner can move)

def feature_tokens(feat):
    """Human-readable keyword tokens used to detect a feature in free text."""
    toks = set()
    m = re.search(r"Age_(\d+)", feat)
    if m: toks.update({m.group(1), m.group(1) + "s"})
    if "60_Plus" in feat: toks.update({"60", "senior", "elderly"})
    if "Sales" in feat: toks.update({"sales", "revenue"})
    if "Resident" in feat: toks.update({"resident", "residential"})
    if "Working" in feat: toks.update({"working", "worker", "workplace", "office"})
    if "Floating" in feat: toks.update({"floating", "foot traffic", "footfall", "passerby"})
    for d in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Sunday"]:
        if d in feat: toks.add(d.lower())
    if "Weekend" in feat: toks.update({"weekend"})
    m = re.search(r"TimeSlot_([\dA-Za-z\-]+)", feat)
    if m: toks.update({m.group(1).lower(), m.group(1).split("-")[0]})
    if "Male" in feat: toks.add("male")
    if "Female" in feat: toks.update({"female", "women"})
    return toks

# ----------------------------------------------------------------------------
# PART 1: Input grounding (actionability of the signal handed to the LLM)
# ----------------------------------------------------------------------------
print("=" * 78); print("PART 1: INPUT GROUNDING OF THE TWO ORIGINAL CASE STUDIES"); print("=" * 78)
case_rows = []
for i in (1, 2):
    ch = pd.read_csv(f"../data/original_outputs/consulting_case{i}_changes.csv")
    tot = len(ch)
    act = ch["Feature"].isin(ACTIONABLE).sum()
    imm = ch["Feature"].isin(IMMUTABLE).sum()
    case_rows.append({"Case": f"case{i}", "District": ch["trade_area_name"].iloc[0],
                      "N_changes": tot, "Actionable": act, "Immutable": imm,
                      "Input_actionability": act / tot if tot else np.nan})
case_in = pd.DataFrame(case_rows)
print(case_in.to_string(index=False))
print("  -> The Itaewon case feeds the LLM a counterfactual basis that is mostly "
      "non-controllable\n     district demography (e.g. resident/floating population), "
      "not owner-actionable levers.")

print(f"\nThree-tier taxonomy: direct_control={len(DIRECT_CONTROL)} (none in data), "
      f"mediating_outcome={sum(v=='mediating_outcome' for v in TIER.values())}, "
      f"exogenous_context={sum(v=='exogenous_context' for v in TIER.values())}")

# Scaled input-grounding across the full matched NB2 sample (both regimes),
# per store, with bootstrap CIs -- this moves the leakage claim from two hand
# -picked cases to the whole sample. ACT_IDX marks owner-influenceable features.
def boot_ci(x, b=2000, seed=44):
    x = np.asarray(x, float); r = np.random.RandomState(seed)
    s = [np.mean(x[r.randint(0, len(x), len(x))]) for _ in range(b)]
    return float(np.mean(x)), float(np.percentile(s, 2.5)), float(np.percentile(s, 97.5))

gen_path = Path("../results/tables/cf_generated_sample.csv")
if gen_path.exists():
    gen = pd.read_csv(gen_path)
    orig = pd.read_csv("../data/modeling_dataset_top30.csv").set_index("store_id")[features]
    ACT_IDX = [features.index(f) for f in ACTIONABLE]
    recs, dist_rows = [], []
    for (regime, ncf), grp in gen.groupby(["regime", "num_cfs"]):
        # per-store (per counterfactual row) actionable-grounding + leakage flag
        per = []
        for sid, g in grp.groupby("store_id"):
            oa = orig.loc[sid].values
            diff = np.abs(g[features].values - oa) > 1e-6
            tot = diff.sum(); act = diff[:, ACT_IDX].sum()
            ag = act / tot if tot else 1.0                 # actionable-grounding
            leaks = int((diff[:, [features.index(f) for f in IMMUTABLE]].sum(axis=1) > 0).any())
            per.append((ag, leaks))
        ag_vals = [p[0] for p in per]; leak_vals = [p[1] for p in per]
        m, lo, hi = boot_ci(ag_vals)
        recs.append({"regime": regime, "num_cfs": ncf, "n_stores": len(per),
                     "actionable_grounding_mean": round(m, 3),
                     "actionable_grounding_ci": f"[{lo:.3f}, {hi:.3f}]",
                     "leaky_cf_rate": round(np.mean(leak_vals), 3)})
    inp = pd.DataFrame(recs)
    inp.to_csv(TAB_DIR / "t8_scaled_input_grounding.csv", index=False)
    print("\nScaled input grounding across the matched sample "
          "(actionable-grounding = share of CF changes on owner-influenceable features;"
          "\n leaky-CF rate = share of counterfactuals touching >=1 exogenous feature):")
    print(inp.to_string(index=False))
    for regime, g in inp.groupby("regime"):
        print(f"  {regime}: mean actionable-grounding "
              f"{g['actionable_grounding_mean'].mean():.3f}, "
              f"mean leaky-CF rate {g['leaky_cf_rate'].mean():.3f}")

# ----------------------------------------------------------------------------
# PART 2: Offline faithfulness metrics on the delivered strategy texts
# ----------------------------------------------------------------------------
print("\n" + "=" * 78); print("PART 2: OFFLINE FAITHFULNESS METRICS ON STRATEGY TEXTS"); print("=" * 78)
SECTIONS = ["commercial district", "success factor", "strateg", "kpi", "implementation"]

def evaluate_strategy(text, change_df):
    low = text.lower()
    changed = change_df["Feature"].unique().tolist()
    grounded = [f for f in changed if any(tok in low for tok in feature_tokens(f))]
    grounding_cov = len(grounded) / len(changed) if changed else np.nan
    imm_changed = [f for f in changed if f in IMMUTABLE]
    imm_leak = [f for f in imm_changed if any(tok in low for tok in feature_tokens(f))]
    leak_rate = len(imm_leak) / len(imm_changed) if imm_changed else 0.0
    completeness = np.mean([s in low for s in SECTIONS])
    return grounding_cov, leak_rate, completeness, len(changed), len(imm_changed)

frows = []
for i in (1, 2):
    txt = open(f"../data/original_outputs/consulting_case{i}_strategy.txt", encoding="utf-8").read()
    ch = pd.read_csv(f"../data/original_outputs/consulting_case{i}_changes.csv")
    g, leak, comp, nchg, nimm = evaluate_strategy(txt, ch)
    frows.append({"Case": f"case{i}", "District": ch["trade_area_name"].iloc[0],
                  "Changed_features": nchg, "Immutable_in_input": nimm,
                  "Grounding_coverage": g, "Nonactionable_leakage": leak,
                  "Structural_completeness": comp, "Strategy_chars": len(txt)})
faith = pd.DataFrame(frows)
faith.to_csv(TAB_DIR / "t6_strategy_faithfulness.csv", index=False)
print(faith.round(3).to_string(index=False))
print("  Grounding_coverage      : share of counterfactual features the strategy actually discusses")
print("  Nonactionable_leakage   : share of IMMUTABLE input features the strategy treats as advice")
print("  Structural_completeness : share of the 5 mandated strategy sections present")

# ----------------------------------------------------------------------------
# PART 3: Confidence-gating eligibility (circularity mitigation)
# ----------------------------------------------------------------------------
print("\n" + "=" * 78); print("PART 3: CONFIDENCE-GATING ELIGIBILITY"); print("=" * 78)
df = pd.read_csv("../data/modeling_dataset_top30.csv")
X, yv = df[features], df["closure"]
X_tr, X_te, y_tr, y_te = train_test_split(X, yv, test_size=0.2, random_state=42, stratify=yv)
model = xgb.XGBClassifier(learning_rate=0.01, max_depth=5, min_child_weight=3,
                          n_estimators=100, subsample=0.8, random_state=44,
                          eval_metric="logloss").fit(X_tr, y_tr)
proba = model.predict_proba(df[features])[:, 1]
closed_pred = (df["closure"] == 1).values & (proba >= 0.5)

# Only generate strategies for closure instances predicted with a margin above
# the 0.5 boundary; larger margin -> fewer but more trustworthy instances.
rows = []
for margin in [0.00, 0.02, 0.04, 0.06, 0.08, 0.10]:
    elig = closed_pred & (proba >= 0.5 + margin)
    rows.append({"confidence_margin": margin, "eligible_instances": int(elig.sum()),
                 "share_of_closed": elig.sum() / (df["closure"] == 1).sum()})
gate = pd.DataFrame(rows)
gate.to_csv(TAB_DIR / "t7_confidence_gating.csv", index=False)
print(gate.round(3).to_string(index=False))
print(f"  max achievable closure probability under this model: {proba.max():.3f}")
print("  -> A low-accuracy model yields few confidently-predicted closure cases; "
      "strategy\n     generation should be gated to this trustworthy subset.")

# ----------------------------------------------------------------------------
# PART 4: Figures
# ----------------------------------------------------------------------------
print("\n" + "=" * 78); print("PART 4: FIGURES"); print("=" * 78)

# F7: faithfulness metrics per case.
fig, ax = plt.subplots(figsize=(6.6, 4.2))
x = np.arange(len(faith)); w = 0.26
ax.bar(x - w, faith["Grounding_coverage"], width=w, color=GRAY_DARK,
       edgecolor="black", label="Grounding coverage")
ax.bar(x, faith["Nonactionable_leakage"], width=w, color=GRAY_MID,
       edgecolor="black", label="Non-actionable leakage")
ax.bar(x + w, faith["Structural_completeness"], width=w, color=GRAY_LIGHT,
       edgecolor="black", label="Structural completeness")
ax.set_xticks(x); ax.set_xticklabels([f"{r.Case}\n({r.District})" for r in faith.itertuples()])
ax.set_ylabel("Score"); ax.set_ylim(0, 1.1); ax.legend(frameon=True, loc="upper center", ncol=1)
save_fig(fig, "f7_strategy_faithfulness")

# F8: confidence-gating trade-off.
fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.plot(gate["confidence_margin"], gate["eligible_instances"], marker="o",
        color=GRAY_DARK)
ax.set_xlabel("Confidence margin above 0.5 decision boundary")
ax.set_ylabel("Eligible closure instances")
save_fig(fig, "f8_confidence_gating")

# ----------------------------------------------------------------------------
# PART 5: API-gated LLM-as-judge + prompt/model ablation harness
# ----------------------------------------------------------------------------
print("\n" + "=" * 78); print("PART 5: LLM-AS-JUDGE & ABLATION HARNESS (API-gated)"); print("=" * 78)

# Load credentials / model tiers from a local .env if present (never committed).
try:
    from dotenv import load_dotenv
    for p in [Path(".env"), Path("../.env")]:
        if p.exists():
            load_dotenv(p); print(f"  loaded environment from {p}"); break
except ImportError:
    pass

# Three model tiers (overridable via environment variables).
GEN_MODELS = [m for m in [os.getenv("LLM_MODEL_WEAK", "gpt-4o-mini"),
                          os.getenv("LLM_MODEL_MID"),
                          os.getenv("LLM_MODEL_STRONG")] if m]
JUDGE_MODEL = os.getenv("LLM_MODEL_STRONG") or os.getenv("LLM_MODEL_MID") or GEN_MODELS[0]
print(f"  generation models: {GEN_MODELS}")
print(f"  judge model      : {JUDGE_MODEL}")

RUBRIC = {  # 1-5 scale, scored by an LLM judge
    "grounding":     "Does the strategy reference the specific counterfactual feature changes?",
    "actionability": "Are the recommendations within the owner's control (not demographics)?",
    "specificity":   "Are the actions concrete and district-specific rather than generic?",
    "feasibility":   "Are the recommendations realistic in cost and effort?",
    "completeness":  "Does it cover district analysis, success factors, strategy, KPI, and implementation?",
}

def build_messages(service_type, location, change_data, restrict=False):
    """Consulting prompt for one case. When restrict=True, an explicit instruction
    tells the LLM to ignore non-controllable demographic changes and recommend only
    owner-controllable actions -- the prompt-only baseline that tests whether
    prompting alone can fix leakage without constraining the counterfactual."""
    changes_text = "\n".join(
        f"- {c['Feature']}: {c['Original Value']:.2f} -> {c['Counterfactual Value']:.2f}"
        for c in change_data)
    system = ("You are a commercial district consulting expert with over 15 years of "
              "experience. Produce a structured strategy with these sections: "
              "Commercial District Status Analysis; Key Success Factors; Customized "
              "Success Strategies; KPI and Monitoring Plans; Implementation Plans.")
    if restrict:
        system += (" IMPORTANT: Some of the provided changes concern district "
                   "demographics (resident, working, or floating population) that an "
                   "individual owner CANNOT control. Do NOT recommend changing those; "
                   "base every recommendation only on owner-controllable levers "
                   "(menu, pricing, hours, promotion, targeting, service).")
    user = (f"Develop a consulting strategy for a prospective entrepreneur opening a "
            f"[{service_type}] in [{location}]. The DiCE counterfactual changes for "
            f"'Closure -> Operating' are:\n{changes_text}\n"
            f"Base your recommendations only on changes the owner can influence.")
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

def judge_messages(strategy, change_data):
    rubric_text = "\n".join(f"- {k}: {v}" for k, v in RUBRIC.items())
    changes_text = ", ".join(c["Feature"] for c in change_data)
    system = ("You are a strict evaluator. Score the consulting strategy on each "
              "criterion from 1 (poor) to 5 (excellent). Return a JSON object with "
              "one integer per criterion key and no other text.")
    user = (f"Counterfactual features: {changes_text}\n\nCriteria:\n{rubric_text}\n\n"
            f"Strategy:\n{strategy}")
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

def run_llm_evaluation(models=None, judge_model=None,
                       prompt_variants=("constrained", "unconstrained", "unconstrained_promptfix")):
    """Full generation + LLM-as-judge pipeline. Requires OPENAI_API_KEY.

    For every (case x prompt_variant x generation model) the pipeline generates a
    strategy and scores it with a fixed judge model on the 5-point rubric.
    `prompt_variant` controls whether the counterfactual basis is filtered to the
    actionable features (constrained) or left as-is (unconstrained), giving the
    key (2)x(4) contrast across the three model tiers at once.
    """
    from langchain_openai import ChatOpenAI
    from langchain_community.adapters.openai import convert_openai_messages
    import json
    models = models or GEN_MODELS
    judge_model = judge_model or JUDGE_MODEL
    out = []
    for i in (1, 2):
        ch = pd.read_csv(f"../data/original_outputs/consulting_case{i}_changes.csv")
        base = {"service_type": ch["service_type"].iloc[0], "location": ch["trade_area_name"].iloc[0]}
        for variant in prompt_variants:
            # constrained: filter CF basis to actionable features (our method).
            # unconstrained: full CF basis, standard prompt.
            # unconstrained_promptfix: full CF basis, but prompt forbids
            #   non-controllable advice (prompt-only baseline).
            sub = ch[ch["Feature"].isin(ACTIONABLE)] if variant == "constrained" else ch
            restrict = (variant == "unconstrained_promptfix")
            cdata = sub[["Feature", "Original Value", "Counterfactual Value"]].to_dict("records")
            if not cdata:
                continue
            for mdl in models:
                try:
                    gen = ChatOpenAI(model=mdl, temperature=0.2, max_tokens=1500)
                    strat = gen.invoke(convert_openai_messages(
                        build_messages(base["service_type"], base["location"], cdata,
                                       restrict=restrict))).content
                    judge = ChatOpenAI(model=judge_model, temperature=0.0, max_tokens=300)
                    raw = judge.invoke(convert_openai_messages(judge_messages(strat, cdata))).content
                    scores = json.loads(re.search(r"\{.*\}", raw, re.S).group(0))
                except Exception as e:
                    scores = {k: np.nan for k in RUBRIC}; scores["error"] = str(e)[:120]
                # offline cross-check: non-actionable leakage of the generated text
                g, leak, comp, _, _ = evaluate_strategy(
                    strat if "strat" in dir() else "", sub)
                out.append({"case": f"case{i}", "variant": variant, "gen_model": mdl,
                            "judge_model": judge_model, **scores,
                            "offline_leakage": leak, "offline_completeness": comp})
    df_scores = pd.DataFrame(out)
    df_scores.to_csv(TAB_DIR / "t9_llm_judge_scores.csv", index=False)
    return df_scores

if os.getenv("OPENAI_API_KEY"):
    print("  OPENAI_API_KEY detected -> running LLM generation + judge ablation ...")
    scores = run_llm_evaluation()
    print(scores.round(2).to_string(index=False))
    print("  saved: t9_llm_judge_scores.csv")
else:
    print("  OPENAI_API_KEY not set -> live LLM scoring skipped (offline metrics above stand).")
    print("  To reproduce the rubric scoring and the constrained-vs-unconstrained prompt")
    print("  ablation across the three model tiers, set OPENAI_API_KEY in a local .env")
    print("  and re-run this notebook (run_llm_evaluation() is called automatically).")

print("\n[NB3 DONE] LLM strategy evaluation framework complete.")


PART 1: INPUT GROUNDING OF THE TWO ORIGINAL CASE STUDIES
 Case                      District  N_changes  Actionable  Immutable  Input_actionability
case1 Children's Grand Park Station          2           2          0                  1.0
case2     Itaewon (Itaewon Station)         10           2          8                  0.2
  -> The Itaewon case feeds the LLM a counterfactual basis that is mostly non-controllable
     district demography (e.g. resident/floating population), not owner-actionable levers.

Three-tier taxonomy: direct_control=0 (none in data), mediating_outcome=19, exogenous_context=11

Scaled input grounding across the matched sample (actionable-grounding = share of CF changes on owner-influenceable features;
 leaky-CF rate = share of counterfactuals touching >=1 exogenous feature):
       regime  num_cfs  n_stores  actionable_grounding_mean actionable_grounding_ci  leaky_cf_rate
  constrained        2       120                      1.000          [1.000, 1.000]      

  saved figure: f8_confidence_gating.png / f8_confidence_gating.pdf

PART 5: LLM-AS-JUDGE & ABLATION HARNESS (API-gated)
  generation models: ['gpt-4o-mini']
  judge model      : gpt-4o-mini
  OPENAI_API_KEY not set -> live LLM scoring skipped (offline metrics above stand).
  To reproduce the rubric scoring and the constrained-vs-unconstrained prompt
  ablation across the three model tiers, set OPENAI_API_KEY in a local .env
  and re-run this notebook (run_llm_evaluation() is called automatically).

[NB3 DONE] LLM strategy evaluation framework complete.
